# Eksperimen Sensitivitas Seluruh Parameter -- ARIMA & LSTM (Apple-to-Apple)

Notebook ini menguji **setiap parameter yang dipakai pipeline** secara empiris menggunakan
pendekatan **One-Factor-At-a-Time (OFAT)**: satu parameter diubah-ubah pada beberapa nilai
kandidat, sementara parameter lain dikunci pada nilai baseline (konfigurasi final skripsi),
lalu MAE/RMSE ARIMA dan LSTM dibandingkan untuk menentukan nilai terbaik secara berbasis bukti.

**Kenapa OFAT, bukan grid search penuh?** Grid search penuh atas 9 parameter dengan rata-rata
4 kandidat per parameter berarti 4^9 (>260.000) kombinasi -- mustahil dijalankan untuk skripsi
mengingat satu kali training penuh (ARIMA 166 produk + LSTM 5 klaster) memakan waktu puluhan
menit hingga lebih dari 1 jam tergantung parameter. OFAT adalah kompromi standar dalam riset
terapan: setiap parameter diuji independen terhadap baseline yang sama, sehingga tetap
memberikan bukti kuantitatif untuk tiap keputusan parameter tanpa biaya komputasi yang
tidak realistis. Keterbatasan ini (potensi interaksi antar-parameter tidak tertangkap)
dicantumkan sebagai batasan penelitian di Bab 5.

**Parameter yang diuji (9 total):**

| No | Parameter | Peran | Kandidat yang diuji |
|---|---|---|---|
| 1 | `SPLIT_PCT` | Rasio bulan training vs testing | 0.60, 0.70, 0.75, 0.80, 0.85 |
| 2 | `MIN_BULAN` | Riwayat minimal produk layak dimodelkan | 10, 12, 15, 18, 20, 24 |
| 3 | `N_WINDOW_ARIMA` | Jendela bulan terakhir yang dipakai ARIMA | 12, 18, 24, 30, 36 |
| 4 | `SEQ_LEN` | Panjang jendela sequence LSTM | 3, 6, 9, 12 |
| 5 | `N_CLUSTER` | Jumlah klaster volume produk utk LSTM | 3, 4, 5, 6, 7 |
| 6 | `IQR_MULTIPLIER` | Faktor pengali batas outlier IQR | 1.0, 1.5, 2.0, 2.5 |
| 7 | `BIAS_HOLDOUT` | Jumlah bulan hold-out utk koreksi bias ARIMA | 4, 6, 7, 9 |
| 8 | `CAP_FACTOR_ARIMA` | Batas atas prediksi ARIMA relatif thd rekor historis | 0.8, 1.0, 1.2, 1.5 |
| 9 | `MIN_BULAN_AKTIF` | Jumlah bulan terakhir utk cek keaktifan produk | 2, 3, 4, 6 |

Parameter yang **TIDAK diuji** dan alasannya:
- `RANDOM_SEED=42` -- bukan hyperparameter yang dicari "terbaik", melainkan kebutuhan
  reproducibility; nilai spesifiknya tidak memengaruhi kualitas model secara sistematis.
- `CAP_FACTOR_LSTM=1.5` -- mengikuti pola yang sama dengan CAP_FACTOR_ARIMA (poin 8); untuk
  menghemat waktu eksperimen, cukup salah satu cap factor yang diuji secara mendalam dan
  hasilnya dijadikan acuan proporsional untuk yang lain.
- `ARIMA_ORDERS` (grid 12 kombinasi p,d,q) -- ini adalah *ruang pencarian* itu sendiri
  (dipilihkan otomatis oleh AIC), bukan satu nilai tunggal yang bisa "diuji terbaik/terburuk"
  seperti parameter skalar lainnya.
- `KALENDER_LIBUR`, `BULAN_EKSKLUDE` -- bukan hyperparameter statistik, melainkan pengetahuan
  domain (kalender libur nasional aktual) yang nilainya sudah given oleh fakta historis,
  bukan sesuatu yang "dioptimasi".
- Hyperparameter arsitektur LSTM (unit 64/32, dropout 0.15, learning rate 3e-4, batch size 16,
  patience EarlyStopping) -- mengikuti praktik umum deep learning yang risikonya sudah
  dimitigasi EarlyStopping + ReduceLROnPlateau; grid search penuh pada parameter ini
  membutuhkan puluhan run tambahan yang tidak proporsional dengan manfaatnya untuk skripsi.

> Catatan eksekusi: notebook ini membutuhkan `dataset_toko.csv`. Setiap eksperimen pada
> notebook ini menjalankan pipeline penuh (training ARIMA + LSTM), sehingga **total waktu
> eksekusi bisa berjam-jam** tergantung spesifikasi komputer. Disarankan menjalankan per
> parameter secara terpisah (jalankan Sel 0-2 sekali, lalu jalankan blok eksperimen satu per
> satu sesuai kebutuhan), bukan menjalankan seluruh notebook sekaligus dari atas ke bawah.

## Sel 0 -- Setup Environment dan Nilai Baseline

In [1]:
import os, random, time, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ.setdefault('PYTHONHASHSEED', '0')

import pandas as pd
import numpy as np
import tensorflow as tf
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

pd.set_option('display.max_columns', None)
csv_path = "dataset_toko.csv"

# ===== NILAI BASELINE (konfigurasi final skripsi) =====
BASELINE = dict(
    SPLIT_PCT=0.80, MIN_BULAN=15, N_WINDOW_ARIMA=24, SEQ_LEN=6, N_CLUSTER=5,
    IQR_MULTIPLIER=1.5, BIAS_HOLDOUT=7, CAP_FACTOR_ARIMA=1.0, CAP_FACTOR_LSTM=1.5,
    MIN_BULAN_AKTIF=3, RANDOM_SEED=42,
)

ARIMA_ORDERS = [
    (1,1,1),(1,1,0),(0,1,1),(2,1,0),
    (0,1,2),(2,1,1),(1,1,2),(3,1,0),
    (0,1,3),(2,1,2),(1,2,1),(0,2,1),
]
KALENDER_LIBUR = {
    '2020-05':0.40,'2021-05':0.45,'2022-05':0.55,'2023-04':0.50,'2024-04':0.65,'2025-03':0.90,
    '2020-06':0.60,'2021-06':0.60,'2022-06':0.65,'2020-07':0.60,'2021-07':0.65,'2022-07':0.70,
    '2023-06':0.65,'2024-05':0.60,'2024-06':0.12,'2024-07':0.28,'2024-08':0.55,'2024-12':0.40,'2025-01':0.58,
}
BULAN_EKSKLUDE = ['2024-06','2024-07']
BULAN_ANOMALI  = BULAN_EKSKLUDE
FEATURES = ['lag1_r','lag2_r','lag3_r','lag6_r','lag12_r',
            'roll3_r','roll6_r','tren_3m','bulan','produk_id','faktor_libur']

def set_seed_ulang(seed, offset=0):
    random.seed(seed + offset)
    np.random.seed(seed + offset)
    tf.random.set_seed(seed + offset)

print("Baseline:", BASELINE)

Baseline: {'SPLIT_PCT': 0.8, 'MIN_BULAN': 15, 'N_WINDOW_ARIMA': 24, 'SEQ_LEN': 6, 'N_CLUSTER': 5, 'IQR_MULTIPLIER': 1.5, 'BIAS_HOLDOUT': 7, 'CAP_FACTOR_ARIMA': 1.0, 'CAP_FACTOR_LSTM': 1.5, 'MIN_BULAN_AKTIF': 3, 'RANDOM_SEED': 42}


## Sel 1 -- Fungsi Helper (parameterized: menerima dict `p` berisi nilai parameter aktif)

In [2]:
def hitung_momentum(df_c):
    if len(df_c) < 3:
        return 1.0
    vals = df_c['qty'].values[-3:]
    if vals[0] > 0:
        tren = (vals[-1] - vals[0]) / vals[0]
        return float(np.clip(1.0 + tren * 0.10, 0.90, 1.10))
    return 1.0

def prediksi_arima(p, bulan_pred, monthly_avail, produk_layak, best_orders_arima, bias_correction, harga_rata2, max_qty_produk):
    bulan_str = str(bulan_pred)
    faktor_libur = KALENDER_LIBUR.get(bulan_str, 1.0)
    hasil = []
    for produk in produk_layak:
        df_p = monthly_avail[monthly_avail['Nama Produk'] == produk].sort_values('bulan_period')
        df_c = df_p[~df_p['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)]
        if len(df_c) > p['N_WINDOW_ARIMA']:
            df_c = df_c.tail(p['N_WINDOW_ARIMA'])
        fc = None
        if produk in best_orders_arima and len(df_c) >= 6:
            try:
                qty_vals = df_c['qty'].clip(lower=0.1).values
                fl_vals  = df_c['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x, 1.0)).values
                ts_log   = np.log1p(qty_vals / np.maximum(fl_vals, 0.1))
                m = ARIMA(ts_log, order=best_orders_arima[produk]).fit()
                fc_result = m.forecast(steps=1)
                fc_val = fc_result.iloc[0] if hasattr(fc_result,'iloc') else np.asarray(fc_result).reshape(-1)[0]
                fc = max(0.0, float(np.expm1(float(np.clip(fc_val,-2.0,10.0)))) * faktor_libur)
                bias = bias_correction.get(produk, 1.0)
                if bias > 0: fc = fc / bias
                fc = fc * hitung_momentum(df_c)
                fc = min(max(0.0, fc), max_qty_produk.get(produk, fc) * p['CAP_FACTOR_ARIMA'])
            except Exception:
                fc = None
        if fc is None and len(df_c) >= 1:
            recent = df_c['qty'].values[-min(6,len(df_c)):]
            bobot  = np.array([1,2,3,4,5,6][-len(recent):], dtype=float)
            fc     = float(np.average(recent, weights=bobot)) * faktor_libur
        if fc is None:
            fc = float(df_c['qty'].median()) * faktor_libur if len(df_c) > 0 else 0.0
        hasil.append({'nama_produk':produk,'pred_qty_arima':round(max(0.0,fc),2),
                      'pred_rev_arima':round(max(0.0,fc)*harga_rata2.get(produk,0),0)})
    return pd.DataFrame(hasil) if hasil else pd.DataFrame(columns=['nama_produk','pred_qty_arima','pred_rev_arima'])

def prediksi_lstm(p, bulan_pred, monthly_avail, produk_layak, cluster_models, cluster_scalers,
                   produk_cluster, le, harga_rata2, median_qty_produk, max_qty_produk):
    bulan_str = str(bulan_pred)
    faktor_libur = KALENDER_LIBUR.get(bulan_str, 1.0)
    bulan_int = bulan_pred.month
    hasil = []
    SEQ_LEN = p['SEQ_LEN']
    for produk in produk_layak:
        if produk not in le.classes_: continue
        cid = produk_cluster.get(produk, 0)
        if cid not in cluster_models or cid not in cluster_scalers: continue
        df_p = monthly_avail[monthly_avail['Nama Produk'] == produk].sort_values('bulan_period')
        df_c = df_p[~df_p['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)]
        if len(df_c) < SEQ_LEN + 4: continue
        med_q = median_qty_produk.get(produk, 1.0)
        qty_v = df_c['qty'].values; n = len(qty_v)
        seq_feats = []
        for step in range(SEQ_LEN):
            idx_cur = n - 1 - (SEQ_LEN - 1 - step)
            if idx_cur < 0: break
            def lag(k): return qty_v[max(idx_cur-k,0)] / max(med_q,1.0)
            r3 = float(np.mean(qty_v[max(0,idx_cur-3):idx_cur])) / max(med_q,1.0) if idx_cur >= 3 else lag(1)
            r6 = float(np.mean(qty_v[max(0,idx_cur-6):idx_cur])) / max(med_q,1.0) if idx_cur >= 6 else r3
            tren = ((qty_v[idx_cur]-qty_v[max(0,idx_cur-3)])/max(abs(qty_v[max(0,idx_cur-3)]),1.0)) if idx_cur>=3 else 0.0
            try:
                curr_bp = df_c['bulan_period'].iloc[idx_cur]
                bulan_step = curr_bp.month; fl_step = KALENDER_LIBUR.get(str(curr_bp), 1.0)
            except Exception:
                bulan_step = bulan_int; fl_step = faktor_libur
            seq_feats.append([lag(1),lag(2),lag(3),lag(6),lag(12),r3,r6,tren,bulan_step,
                              float(le.transform([produk])[0]),fl_step])
        if len(seq_feats) < SEQ_LEN: continue
        seq_sc = cluster_scalers[cid].transform(np.array(seq_feats[-SEQ_LEN:])).reshape(1,SEQ_LEN,len(FEATURES))
        pred_r = float(np.expm1(float(np.clip(cluster_models[cid].predict(seq_sc,verbose=0)[0,0],0,None))))
        pred_qty = min(max(0.0,max(0.0,pred_r*med_q)*faktor_libur), max_qty_produk.get(produk,pred_r*med_q)*p['CAP_FACTOR_LSTM'])
        hasil.append({'nama_produk':produk,'pred_qty_lstm':round(pred_qty,2),
                      'pred_rev_lstm':round(pred_qty*harga_rata2.get(produk,0),0)})
    return pd.DataFrame(hasil) if hasil else pd.DataFrame(columns=['nama_produk','pred_qty_lstm','pred_rev_lstm'])

print("Fungsi helper (parameterized) siap.")

Fungsi helper (parameterized) siap.


## Sel 2 -- Fungsi Utama: `jalankan_pipeline(overrides)`

Menjalankan seluruh pipeline (preprocessing -> IQR -> filter produk -> training ARIMA -> training LSTM -> evaluasi rolling) untuk SATU kombinasi parameter, dan mengembalikan MAE/RMSE ARIMA & LSTM. Parameter yang tidak disebut di `overrides` memakai nilai BASELINE.

In [3]:
def jalankan_pipeline(overrides, label_eksperimen=""):
    p = {**BASELINE, **overrides}
    set_seed_ulang(p['RANDOM_SEED'])
    print(f"\n{'='*70}\n[EKSPERIMEN] {label_eksperimen} -> {overrides}\n{'='*70}")

    df = pd.read_csv(csv_path, on_bad_lines='skip')
    df['Tanggal Pembayaran'] = pd.to_datetime(df['Tanggal Pembayaran'], format='mixed', errors='coerce')
    df = df.dropna(subset=['Tanggal Pembayaran'])
    df = df[df['Status Terakhir'] == 'Pesanan Selesai'].copy()
    for col in ['Harga Jual (IDR)','Jumlah Produk Dibeli']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['item_revenue'] = (df['Harga Jual (IDR)'] * df['Jumlah Produk Dibeli']).clip(lower=0)
    df['bulan_period'] = df['Tanggal Pembayaran'].dt.to_period('M')
    df['bulan']        = df['Tanggal Pembayaran'].dt.month
    harga_rata2 = df.groupby('Nama Produk')['Harga Jual (IDR)'].mean().to_dict()

    bulan_list  = sorted(df['bulan_period'].unique())
    split_idx   = int(len(bulan_list) * p['SPLIT_PCT'])
    bulan_train = bulan_list[:split_idx]
    bulan_test  = bulan_list[split_idx:]

    monthly_all = (df.groupby(['bulan_period','bulan','Nama Produk'])
                     .agg(qty=('Jumlah Produk Dibeli','sum'), revenue=('item_revenue','sum'))
                     .reset_index().sort_values(['Nama Produk','bulan_period']))
    monthly_train = monthly_all[monthly_all['bulan_period'].isin(bulan_train)]
    monthly_test  = monthly_all[monthly_all['bulan_period'].isin(bulan_test)]

    # IQR clipping leak-free
    iqr_bounds = {}
    for prod in monthly_train['Nama Produk'].unique():
        vals = monthly_train[monthly_train['Nama Produk']==prod]['qty']
        if len(vals) < 4: continue
        Q1, Q3 = vals.quantile(0.25), vals.quantile(0.75)
        IQR = Q3 - Q1
        iqr_bounds[prod] = (max(0.0, Q1-p['IQR_MULTIPLIER']*IQR), Q3+p['IQR_MULTIPLIER']*IQR)
    def _clip_qty(row):
        b = iqr_bounds.get(row['Nama Produk'])
        return row['qty'] if b is None else float(np.clip(row['qty'], b[0], b[1]))
    monthly_all['qty'] = monthly_all.apply(_clip_qty, axis=1)
    monthly_train = monthly_all[monthly_all['bulan_period'].isin(bulan_train)]
    monthly_test  = monthly_all[monthly_all['bulan_period'].isin(bulan_test)]

    # Filter produk layak (apple-to-apple)
    produk_count = monthly_train.groupby('Nama Produk')['bulan_period'].count()
    produk_layak_awal = produk_count[produk_count >= p['MIN_BULAN']].index.tolist()
    bulan_train_bersih = [b for b in bulan_train if str(b) not in BULAN_ANOMALI]
    bulan_cek_aktif = bulan_train_bersih[-p['MIN_BULAN_AKTIF']:]
    def cek_aktif(prod):
        df_p = monthly_train[monthly_train['Nama Produk']==prod]
        return df_p[df_p['bulan_period'].isin(bulan_cek_aktif)]['qty'].sum() > 0
    produk_layak = [prod for prod in produk_layak_awal if cek_aktif(prod)]
    produk_arima_l = list(produk_layak)

    if len(produk_layak) < 5:
        print(f"[SKIP] Hanya {len(produk_layak)} produk layak -- terlalu sedikit, eksperimen dilewati.")
        return None

    mean_qty_produk, median_qty_produk, max_qty_produk = {}, {}, {}
    for prod in produk_layak:
        vals = monthly_train[(monthly_train['Nama Produk']==prod) & (~monthly_train['bulan_period'].astype(str).isin(BULAN_ANOMALI))]['qty'].values
        mean_qty_produk[prod]   = max(float(vals.mean()), 1.0) if len(vals)>0 else 1.0
        median_qty_produk[prod] = max(float(np.median(vals)), 1.0) if len(vals)>0 else 1.0
        max_qty_produk[prod]    = max(float(vals.max()), 1.0) if len(vals)>0 else 1.0
    print(f"  Produk layak: {len(produk_layak)} | Bulan training: {len(bulan_train)} | Bulan testing: {len(bulan_test)}")

    # Training ARIMA
    t0 = time.time()
    best_orders_arima, bias_correction = {}, {}
    for produk in produk_arima_l:
        df_c = monthly_train[monthly_train['Nama Produk']==produk].sort_values('bulan_period')
        df_c = df_c[~df_c['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)].tail(p['N_WINDOW_ARIMA'])
        if len(df_c) < 10:
            bias_correction[produk] = 1.0
            continue
        fl_v = df_c['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x,1.0)).values
        ts_log = np.log1p(df_c['qty'].clip(lower=0.1).values / np.maximum(fl_v, 0.1))
        best_aic, best_order = np.inf, (1,1,1)
        for order in ARIMA_ORDERS:
            try:
                m = ARIMA(ts_log, order=order).fit()
                if m.aic < best_aic:
                    best_aic, best_order = m.aic, order
            except Exception:
                pass
        best_orders_arima[produk] = best_order
        bias_correction[produk] = 1.0
        if len(df_c) > p['BIAS_HOLDOUT'] + 8:
            try:
                tr_b = df_c.iloc[:-p['BIAS_HOLDOUT']]; ho_b = df_c.iloc[-p['BIAS_HOLDOUT']:]
                ts_tb = np.log1p(tr_b['qty'].clip(lower=0.1).values / np.maximum(
                    tr_b['bulan_period'].astype(str).map(lambda x:KALENDER_LIBUR.get(x,1.0)).values, 0.1))
                m_b = ARIMA(ts_tb, order=best_order).fit()
                fc_log = np.clip(m_b.forecast(steps=p['BIAS_HOLDOUT']), -2.0, 10.0)
                fl_hb  = ho_b['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x,1.0)).values
                fc_qty = np.expm1(fc_log) * fl_hb
                rasio  = max(float(np.asarray(fc_qty).sum()), 1e-6) / float(ho_b['qty'].values.sum())
                bias_correction[produk] = float(np.clip(rasio, 0.7, 1.5))
            except Exception:
                bias_correction[produk] = 1.0
    t_arima = time.time() - t0
    print(f"  ARIMA: {len(best_orders_arima)} produk dilatih dlm {t_arima:.1f} detik")

    # Rekayasa fitur + klasterisasi LSTM
    sorted_prods = sorted(produk_layak, key=lambda prod: median_qty_produk[prod])
    cluster_size = max(1, len(sorted_prods) // p['N_CLUSTER'])
    produk_cluster = {prod: min(i // cluster_size, p['N_CLUSTER']-1) for i, prod in enumerate(sorted_prods)}
    le = LabelEncoder()
    mtr = monthly_train[monthly_train['Nama Produk'].isin(produk_layak)].copy()
    mtr['produk_id'] = le.fit_transform(mtr['Nama Produk'])
    mtr['median_qty'] = mtr['Nama Produk'].map(median_qty_produk)
    mtr['cluster_id'] = mtr['Nama Produk'].map(produk_cluster)
    for lag in [1,2,3,6,12]:
        mtr[f'lag{lag}_r'] = mtr.groupby('Nama Produk', sort=True)['qty'].transform(lambda x: x.shift(lag)) / mtr['median_qty'].clip(lower=1.0)
    for w, col in [(3,'roll3_r'),(6,'roll6_r')]:
        mtr[col] = mtr.groupby('Nama Produk', sort=True)['qty'].transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean()) / mtr['median_qty'].clip(lower=1.0)
    mtr['tren_3m'] = mtr.groupby('Nama Produk', sort=True)['qty'].transform(
        lambda x: x.shift(1).rolling(3, min_periods=2).apply(lambda v: (v[-1]-v[0])/max(abs(v[0]),1), raw=True))
    mtr['faktor_libur'] = mtr['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x, 1.0))
    mtr['log_rasio'] = np.log1p((mtr['qty']/mtr['median_qty'].clip(lower=1.0)).clip(0, 8.0))
    mtr_clean = mtr.dropna()
    mtr_clean = mtr_clean[~mtr_clean['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)].copy()
    mtr_clean = mtr_clean.sort_values(['cluster_id','Nama Produk','bulan_period']).reset_index(drop=True)

    # Training LSTM per klaster
    t0 = time.time()
    cluster_models, cluster_scalers = {}, {}
    SEQ_LEN = p['SEQ_LEN']
    for cid in range(p['N_CLUSTER']):
        df_cl = mtr_clean[mtr_clean['cluster_id']==cid].reset_index(drop=True)
        if len(df_cl) < 40: continue
        set_seed_ulang(p['RANDOM_SEED'], cid)
        scaler = MinMaxScaler()
        X_sc = scaler.fit_transform(df_cl[FEATURES].values.astype(float))
        y_all = df_cl['log_rasio'].values
        X_seq, y_seq = [], []
        for _, grp in df_cl.groupby('Nama Produk', sort=True):
            local_idx = list(grp.sort_values('bulan_period').index)
            for i in range(len(local_idx)-SEQ_LEN):
                X_seq.append(X_sc[local_idx[i:i+SEQ_LEN]])
                y_seq.append(y_all[local_idx[i+SEQ_LEN]])
        if len(X_seq) < 20: continue
        X_3d, y_arr = np.array(X_seq), np.array(y_seq)
        model = Sequential([LSTM(64, input_shape=(SEQ_LEN,len(FEATURES)), return_sequences=True),
                            BatchNormalization(), Dropout(0.15),
                            LSTM(32, return_sequences=False),
                            BatchNormalization(), Dropout(0.15),
                            Dense(16, activation='relu'), Dense(1)])
        model.compile(loss=tf.keras.losses.Huber(delta=1.0), optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4))
        model.fit(X_3d, y_arr, epochs=500, batch_size=16, validation_split=0.2, shuffle=False,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True),
                       tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=15, factor=0.5, min_lr=1e-6, verbose=0)],
            verbose=0)
        cluster_models[cid]=model; cluster_scalers[cid]=scaler
    t_lstm = time.time() - t0
    print(f"  LSTM: {len(cluster_models)} klaster dilatih dlm {t_lstm:.1f} detik")

    # Evaluasi rolling
    log_bulan = []
    for bulan_pred in bulan_test:
        avail = monthly_all[monthly_all['bulan_period'] < bulan_pred]
        df_a = prediksi_arima(p, bulan_pred, avail, produk_layak, best_orders_arima, bias_correction, harga_rata2, max_qty_produk)
        df_l = prediksi_lstm(p, bulan_pred, avail, produk_layak, cluster_models, cluster_scalers, produk_cluster, le,
                              harga_rata2, median_qty_produk, max_qty_produk)
        df_m = pd.merge(df_a, df_l, on='nama_produk', how='outer').fillna(0)
        aktual_b = monthly_test[monthly_test['bulan_period']==bulan_pred][['Nama Produk','qty','revenue']].rename(
            columns={'Nama Produk':'nama_produk','qty':'aktual_qty','revenue':'aktual_rev'})
        df_m = pd.merge(df_m, aktual_b, on='nama_produk', how='left').fillna(0)
        log_bulan.append({'aktual': df_m['aktual_rev'].sum(), 'pred_arima': df_m['pred_rev_arima'].sum(),
                           'pred_lstm': df_m['pred_rev_lstm'].sum()})

    df_log = pd.DataFrame(log_bulan)
    mae_a = float(mean_absolute_error(df_log['aktual'], df_log['pred_arima']))
    rmse_a = float(np.sqrt(mean_squared_error(df_log['aktual'], df_log['pred_arima'])))
    mae_l = float(mean_absolute_error(df_log['aktual'], df_log['pred_lstm']))
    rmse_l = float(np.sqrt(mean_squared_error(df_log['aktual'], df_log['pred_lstm'])))
    print(f"  HASIL -> ARIMA: MAE=Rp{mae_a:,.0f} RMSE=Rp{rmse_a:,.0f} | LSTM: MAE=Rp{mae_l:,.0f} RMSE=Rp{rmse_l:,.0f}")

    return {**overrides, 'label': label_eksperimen, 'jumlah_produk': len(produk_layak),
            'bulan_testing': len(bulan_test),
            'mae_arima': mae_a, 'rmse_arima': rmse_a, 'waktu_arima_detik': round(t_arima,1),
            'mae_lstm': mae_l, 'rmse_lstm': rmse_l, 'waktu_lstm_detik': round(t_lstm,1)}


def rangkum_dan_pilih(hasil_list, nama_parameter):
    df_hasil = pd.DataFrame([h for h in hasil_list if h is not None])
    if df_hasil.empty:
        print(f"[PERINGATAN] Tidak ada hasil valid untuk {nama_parameter}.")
        return df_hasil
    df_hasil['rank_arima'] = df_hasil['mae_arima'].rank()
    df_hasil['rank_lstm']  = df_hasil['mae_lstm'].rank()
    df_hasil['rank_gabungan'] = df_hasil['rank_arima'] + df_hasil['rank_lstm']
    display(df_hasil.drop(columns=['label']))
    terbaik = df_hasil.loc[df_hasil['rank_gabungan'].idxmin()]
    print(f"[REKOMENDASI] {nama_parameter} = {terbaik[nama_parameter]} memberi hasil gabungan terbaik "
          f"(MAE ARIMA=Rp{terbaik['mae_arima']:,.0f}, MAE LSTM=Rp{terbaik['mae_lstm']:,.0f}, "
          f"{int(terbaik['jumlah_produk'])} produk).")
    return df_hasil

print("jalankan_pipeline() dan rangkum_dan_pilih() siap dipakai untuk seluruh eksperimen di bawah.")

jalankan_pipeline() dan rangkum_dan_pilih() siap dipakai untuk seluruh eksperimen di bawah.


## 1. Eksperimen SPLIT_PCT (Rasio Bulan Training vs Testing)

**Alasan pengujian:** Menentukan proporsi bulan yang dipakai belajar (training) vs diuji (testing). Rasio terlalu kecil (training sedikit) membuat model kekurangan data historis; rasio terlalu besar (testing sedikit) membuat evaluasi tidak representatif krn periode testing terlalu pendek untuk mencakup variasi musiman satu tahun penuh.

**Kandidat yang diuji:** [0.6, 0.7, 0.75, 0.8, 0.85]

In [4]:
hasil_split_pct = []
for nilai in [0.6, 0.7, 0.75, 0.8, 0.85]:
    hasil_split_pct.append(jalankan_pipeline({'SPLIT_PCT': nilai}, label_eksperimen='SPLIT_PCT'))

df_split_pct = rangkum_dan_pilih(hasil_split_pct, 'SPLIT_PCT')


[EKSPERIMEN] SPLIT_PCT -> {'SPLIT_PCT': 0.6}
  Produk layak: 151 | Bulan training: 35 | Bulan testing: 24


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 151 produk dilatih dlm 7225.9 detik
  LSTM: 5 klaster dilatih dlm 237.1 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

...

  ARIMA: 163 produk dilatih dlm 574.7 detik
  LSTM: 5 klaster dilatih dlm 159.8 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp1,979,994 RMSE=Rp2,382,254 | LSTM: MAE=Rp1,795,308 RMSE=Rp2,232,838


,SPLIT_PCT,jumlah_produk,bulan_testing,mae_arima,rmse_arima,waktu_arima_detik,mae_lstm,rmse_lstm,waktu_lstm_detik,rank_arima,rank_lstm,rank_gabungan
0,0.60,151,24,4.187069e+06,5.243509e+06,7225.9,4.916342e+06,5.925027e+06,237.1,5.0,5.0,10.0
1,0.70,163,18,3.179438e+06,3.758221e+06,1360.5,3.540218e+06,4.326719e+06,125.8,4.0,4.0,8.0
2,0.75,170,15,2.194101e+06,2.742875e+06,1860.3,3.367361e+06,3.892785e+06,310.7,2.0,3.0,5.0
3,0.80,166,12,2.467634e+06,2.950422e+06,747.7,1.327530e+06,1.587740e+06,159.8,3.0,1.0,4.0
4,0.85,163,9,1.979994e+06,2.382254e+06,574.7,1.795308e+06,2.232838e+06,159.8,1.0,2.0,3.0


[REKOMENDASI] SPLIT_PCT = 0.85 memberi hasil gabungan terbaik (MAE ARIMA=Rp1,979,994, MAE LSTM=Rp1,795,308, 163 produk).


## 2. Eksperimen MIN_BULAN (Riwayat Minimal Produk Layak Dimodelkan)

**Alasan pengujian:** Menentukan ambang riwayat minimal supaya produk diikutsertakan dalam pemodelan. Ambang rendah = lebih banyak produk tercakup tapi riwayat pendek berisiko estimasi tidak stabil (khususnya utk ARIMA yg melatih 1 model per produk secara independen); ambang tinggi = estimasi lebih stabil tapi cakupan produk berkurang.

**Kandidat yang diuji:** [10, 12, 15, 18, 20, 24]

In [5]:
hasil_min_bulan = []
for nilai in [10, 12, 15, 18, 20, 24]:
    hasil_min_bulan.append(jalankan_pipeline({'MIN_BULAN': nilai}, label_eksperimen='MIN_BULAN'))

df_min_bulan = rangkum_dan_pilih(hasil_min_bulan, 'MIN_BULAN')


[EKSPERIMEN] MIN_BULAN -> {'MIN_BULAN': 10}
  Produk layak: 191 | Bulan training: 47 | Bulan testing: 12


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 191 produk dilatih dlm 819.9 detik
  LSTM: 5 klaster dilatih dlm 172.9 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp3,040,348 RMSE=Rp3,600,653 | LSTM: MAE=Rp3,499,848 RMSE=Rp4,092,034

[EKSPERIMEN] MIN_BULAN -> {'MIN_BULAN': 12}
  Produk layak: 180 | Bulan training: 47 | Bulan testing: 12


...

  ARIMA: 133 produk dilatih dlm 744.1 detik
  LSTM: 5 klaster dilatih dlm 375.5 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp1,942,209 RMSE=Rp2,418,139 | LSTM: MAE=Rp4,056,260 RMSE=Rp4,699,995


,MIN_BULAN,jumlah_produk,bulan_testing,mae_arima,rmse_arima,waktu_arima_detik,mae_lstm,rmse_lstm,waktu_lstm_detik,rank_arima,rank_lstm,rank_gabungan
0,10,191,12,3.040348e+06,3.600653e+06,819.9,3.499848e+06,4.092034e+06,172.9,6.0,5.0,11.0
1,12,180,12,2.836414e+06,3.393345e+06,1096.8,3.389055e+06,3.923137e+06,386.7,5.0,4.0,9.0
2,15,166,12,2.467634e+06,2.950422e+06,878.6,1.327530e+06,1.587740e+06,161.2,4.0,1.0,5.0
3,18,156,12,2.242472e+06,2.719353e+06,514.2,2.464601e+06,2.867906e+06,183.1,3.0,2.0,5.0
4,20,153,12,2.173328e+06,2.645353e+06,906.9,2.602080e+06,3.357528e+06,409.3,2.0,3.0,5.0
5,24,133,12,1.942209e+06,2.418139e+06,744.1,4.056260e+06,4.699995e+06,375.5,1.0,6.0,7.0


[REKOMENDASI] MIN_BULAN = 15 memberi hasil gabungan terbaik (MAE ARIMA=Rp2,467,634, MAE LSTM=Rp1,327,530, 166 produk).


## 3. Eksperimen N_WINDOW_ARIMA (Jendela Bulan Terakhir utk ARIMA)

**Alasan pengujian:** Menentukan berapa bulan ke belakang yang dipakai ARIMA sbg basis fitting model. Jendela pendek = ARIMA lebih responsif thd tren terbaru tapi berisiko kehilangan pola musiman tahunan (butuh min. 12 bulan utk 1 siklus); jendela panjang = menangkap musiman lebih baik tapi kurang responsif thd perubahan pola terbaru & lebih lambat dihitung.

**Kandidat yang diuji:** [12, 18, 24, 30, 36]

In [6]:
hasil_n_window_arima = []
for nilai in [12, 18, 24, 30, 36]:
    hasil_n_window_arima.append(jalankan_pipeline({'N_WINDOW_ARIMA': nilai}, label_eksperimen='N_WINDOW_ARIMA'))

df_n_window_arima = rangkum_dan_pilih(hasil_n_window_arima, 'N_WINDOW_ARIMA')


[EKSPERIMEN] N_WINDOW_ARIMA -> {'N_WINDOW_ARIMA': 12}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 166 produk dilatih dlm 887.3 detik
  LSTM: 5 klaster dilatih dlm 217.6 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,095,406 RMSE=Rp2,623,707 | LSTM: MAE=Rp1,327,530 RMSE=Rp1,587,740

[EKSPERIMEN] N_WINDOW_ARIMA -> {'N_WINDOW_ARIMA': 18}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


...

  ARIMA: 166 produk dilatih dlm 1241.1 detik
  LSTM: 5 klaster dilatih dlm 254.4 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,467,536 RMSE=Rp2,934,582 | LSTM: MAE=Rp1,327,530 RMSE=Rp1,587,740


,N_WINDOW_ARIMA,jumlah_produk,bulan_testing,mae_arima,rmse_arima,waktu_arima_detik,mae_lstm,rmse_lstm,waktu_lstm_detik,rank_arima,rank_lstm,rank_gabungan
0,12,166,12,2.095406e+06,2.623707e+06,887.3,1.327530e+06,1.587740e+06,217.6,1.0,3.0,4.0
1,18,166,12,2.474628e+06,2.985201e+06,667.5,1.327530e+06,1.587740e+06,218.2,4.0,3.0,7.0
2,24,166,12,2.467634e+06,2.950422e+06,839.9,1.327530e+06,1.587740e+06,180.8,3.0,3.0,6.0
3,30,166,12,2.710422e+06,3.271370e+06,632.4,1.327530e+06,1.587740e+06,159.9,5.0,3.0,8.0
4,36,166,12,2.467536e+06,2.934582e+06,1241.1,1.327530e+06,1.587740e+06,254.4,2.0,3.0,5.0


[REKOMENDASI] N_WINDOW_ARIMA = 12 memberi hasil gabungan terbaik (MAE ARIMA=Rp2,095,406, MAE LSTM=Rp1,327,530, 166 produk).


## 4. Eksperimen SEQ_LEN (Panjang Jendela Sequence LSTM)

**Alasan pengujian:** Menentukan berapa langkah waktu (bulan) yang 'dilihat' LSTM sebelum menebak 1 bulan ke depan. SEQ_LEN pendek = lebih banyak sampel sequence terbentuk dari riwayat terbatas tapi konteks historis yang dipelajari lebih sedikit; SEQ_LEN panjang = konteks lebih kaya (misal menangkap pola 1 tahun) tapi jumlah sampel sequence yang bisa dibentuk berkurang.

**Kandidat yang diuji:** [3, 6, 9, 12]

In [7]:
hasil_seq_len = []
for nilai in [3, 6, 9, 12]:
    hasil_seq_len.append(jalankan_pipeline({'SEQ_LEN': nilai}, label_eksperimen='SEQ_LEN'))

df_seq_len = rangkum_dan_pilih(hasil_seq_len, 'SEQ_LEN')


[EKSPERIMEN] SEQ_LEN -> {'SEQ_LEN': 3}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 166 produk dilatih dlm 833.2 detik
  LSTM: 5 klaster dilatih dlm 388.0 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,467,634 RMSE=Rp2,950,422 | LSTM: MAE=Rp2,765,429 RMSE=Rp3,304,570

[EKSPERIMEN] SEQ_LEN -> {'SEQ_LEN': 6}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


...

  ARIMA: 166 produk dilatih dlm 232.8 detik
  LSTM: 5 klaster dilatih dlm 208.4 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,467,634 RMSE=Rp2,950,422 | LSTM: MAE=Rp1,334,908 RMSE=Rp1,603,414


,SEQ_LEN,jumlah_produk,bulan_testing,mae_arima,rmse_arima,waktu_arima_detik,mae_lstm,rmse_lstm,waktu_lstm_detik,rank_arima,rank_lstm,rank_gabungan
0,3,166,12,2.467634e+06,2.950422e+06,833.2,2.765429e+06,3.304570e+06,388.0,2.5,3.0,5.5
1,6,166,12,2.467634e+06,2.950422e+06,803.0,1.327530e+06,1.587740e+06,294.6,2.5,1.0,3.5
2,9,166,12,2.467634e+06,2.950422e+06,322.9,2.878264e+06,3.502037e+06,191.7,2.5,4.0,6.5
3,12,166,12,2.467634e+06,2.950422e+06,232.8,1.334908e+06,1.603414e+06,208.4,2.5,2.0,4.5


[REKOMENDASI] SEQ_LEN = 6 memberi hasil gabungan terbaik (MAE ARIMA=Rp2,467,634, MAE LSTM=Rp1,327,530, 166 produk).


## 5. Eksperimen N_CLUSTER (Jumlah Klaster Volume Produk utk LSTM)

**Alasan pengujian:** Menentukan berapa kelompok volume penjualan dipakai utk melatih model LSTM terpisah. Klaster sedikit = tiap model punya lebih banyak sampel tapi rentang volume dlm 1 klaster terlalu lebar (kurang homogen); klaster banyak = lebih homogen per klaster tapi jumlah sampel per model berkurang & risiko klaster terlalu kecil utk dilatih (<40 sampel di-skip).

**Kandidat yang diuji:** [3, 4, 5, 6, 7]

In [5]:
hasil_n_cluster = []
for nilai in [3, 4, 5, 6, 7]:
    hasil_n_cluster.append(jalankan_pipeline({'N_CLUSTER': nilai}, label_eksperimen='N_CLUSTER'))

df_n_cluster = rangkum_dan_pilih(hasil_n_cluster, 'N_CLUSTER')


[EKSPERIMEN] N_CLUSTER -> {'N_CLUSTER': 3}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 166 produk dilatih dlm 572.9 detik
  LSTM: 3 klaster dilatih dlm 97.7 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,467,634 RMSE=Rp2,950,422 | LSTM: MAE=Rp2,210,202 RMSE=Rp2,884,918

[EKSPERIMEN] N_CLUSTER -> {'N_CLUSTER': 4}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


...

  ARIMA: 166 produk dilatih dlm 137.2 detik
  LSTM: 7 klaster dilatih dlm 203.5 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,467,634 RMSE=Rp2,950,422 | LSTM: MAE=Rp2,922,164 RMSE=Rp3,706,368


,N_CLUSTER,jumlah_produk,bulan_testing,mae_arima,rmse_arima,waktu_arima_detik,mae_lstm,rmse_lstm,waktu_lstm_detik,rank_arima,rank_lstm,rank_gabungan
0,3,166,12,2.467634e+06,2.950422e+06,572.9,2.210202e+06,2.884918e+06,97.7,3.0,4.0,7.0
1,4,166,12,2.467634e+06,2.950422e+06,482.2,2.161630e+06,2.584987e+06,116.0,3.0,3.0,6.0
2,5,166,12,2.467634e+06,2.950422e+06,118.4,1.327530e+06,1.587740e+06,117.6,3.0,2.0,5.0
3,6,166,12,2.467634e+06,2.950422e+06,128.2,9.588162e+05,1.396375e+06,138.3,3.0,1.0,4.0
4,7,166,12,2.467634e+06,2.950422e+06,137.2,2.922164e+06,3.706368e+06,203.5,3.0,5.0,8.0


[REKOMENDASI] N_CLUSTER = 6 memberi hasil gabungan terbaik (MAE ARIMA=Rp2,467,634, MAE LSTM=Rp958,816, 166 produk).


## 6. Eksperimen IQR_MULTIPLIER (Faktor Pengali Batas Outlier)

**Alasan pengujian:** Menentukan seberapa agresif nilai qty ekstrem (transaksi grosir) dipangkas. Nilai kecil (1.0) = pemangkasan lebih agresif, berisiko memangkas variasi wajar; nilai besar (2.5) = lebih longgar, berisiko outlier grosir tetap memengaruhi pola yg dipelajari model. 1.5 adalah konvensi statistik standar (Tukey's fences).

**Kandidat yang diuji:** [1.0, 1.5, 2.0, 2.5]

In [6]:
hasil_iqr_multiplier = []
for nilai in [1.0, 1.5, 2.0, 2.5]:
    hasil_iqr_multiplier.append(jalankan_pipeline({'IQR_MULTIPLIER': nilai}, label_eksperimen='IQR_MULTIPLIER'))

df_iqr_multiplier = rangkum_dan_pilih(hasil_iqr_multiplier, 'IQR_MULTIPLIER')


[EKSPERIMEN] IQR_MULTIPLIER -> {'IQR_MULTIPLIER': 1.0}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 166 produk dilatih dlm 142.7 detik
  LSTM: 5 klaster dilatih dlm 141.0 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,510,964 RMSE=Rp3,010,343 | LSTM: MAE=Rp1,816,592 RMSE=Rp2,203,430

[EKSPERIMEN] IQR_MULTIPLIER -> {'IQR_MULTIPLIER': 1.5}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


...

  ARIMA: 166 produk dilatih dlm 838.7 detik
  LSTM: 5 klaster dilatih dlm 328.1 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,627,031 RMSE=Rp3,094,891 | LSTM: MAE=Rp1,317,354 RMSE=Rp1,505,783


,IQR_MULTIPLIER,jumlah_produk,bulan_testing,mae_arima,rmse_arima,waktu_arima_detik,mae_lstm,rmse_lstm,waktu_lstm_detik,rank_arima,rank_lstm,rank_gabungan
0,1.0,166,12,2.510964e+06,3.010343e+06,142.7,1.816592e+06,2.203430e+06,141.0,2.0,4.0,6.0
1,1.5,166,12,2.467634e+06,2.950422e+06,187.6,1.327530e+06,1.587740e+06,133.3,1.0,3.0,4.0
2,2.0,166,12,2.570682e+06,3.041381e+06,792.3,1.214821e+06,1.529969e+06,329.4,3.0,1.0,4.0
3,2.5,166,12,2.627031e+06,3.094891e+06,838.7,1.317354e+06,1.505783e+06,328.1,4.0,2.0,6.0


[REKOMENDASI] IQR_MULTIPLIER = 1.5 memberi hasil gabungan terbaik (MAE ARIMA=Rp2,467,634, MAE LSTM=Rp1,327,530, 166 produk).


## 7. Eksperimen BIAS_HOLDOUT (Jumlah Bulan Hold-out utk Koreksi Bias ARIMA)

**Alasan pengujian:** Menentukan berapa bulan disisihkan utk mengukur & mengoreksi kecenderungan ARIMA over/under-forecast. Hold-out pendek = estimasi bias lebih cepat didapat tapi rentan dipengaruhi 1-2 bulan anomali acak; hold-out panjang = estimasi bias lebih stabil tapi mengurangi data yang tersisa utk fitting model utama.

**Kandidat yang diuji:** [4, 6, 7, 9]

In [4]:
hasil_bias_holdout = []
for nilai in [4, 6, 7, 9]:
    hasil_bias_holdout.append(jalankan_pipeline({'BIAS_HOLDOUT': nilai}, label_eksperimen='BIAS_HOLDOUT'))

df_bias_holdout = rangkum_dan_pilih(hasil_bias_holdout, 'BIAS_HOLDOUT')


[EKSPERIMEN] BIAS_HOLDOUT -> {'BIAS_HOLDOUT': 4}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 166 produk dilatih dlm 1256.8 detik
  LSTM: 5 klaster dilatih dlm 82.5 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

...

  ARIMA: 166 produk dilatih dlm 177.5 detik
  LSTM: 5 klaster dilatih dlm 121.3 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp1,856,552 RMSE=Rp2,375,557 | LSTM: MAE=Rp1,327,530 RMSE=Rp1,587,740


,BIAS_HOLDOUT,jumlah_produk,bulan_testing,mae_arima,rmse_arima,waktu_arima_detik,mae_lstm,rmse_lstm,waktu_lstm_detik,rank_arima,rank_lstm,rank_gabungan
0,4,166,12,1.811498e+06,2.323175e+06,1256.8,1.327530e+06,1.587740e+06,82.5,1.0,2.5,3.5
1,6,166,12,2.176511e+06,2.672532e+06,932.3,1.327530e+06,1.587740e+06,130.1,3.0,2.5,5.5
2,7,166,12,2.467634e+06,2.950422e+06,190.7,1.327530e+06,1.587740e+06,110.7,4.0,2.5,6.5
3,9,166,12,1.856552e+06,2.375557e+06,177.5,1.327530e+06,1.587740e+06,121.3,2.0,2.5,4.5


[REKOMENDASI] BIAS_HOLDOUT = 4 memberi hasil gabungan terbaik (MAE ARIMA=Rp1,811,498, MAE LSTM=Rp1,327,530, 166 produk).


## 8. Eksperimen CAP_FACTOR_ARIMA (Batas Atas Prediksi ARIMA)

**Alasan pengujian:** Menentukan batas atas prediksi ARIMA relatif thd rekor qty tertinggi historis produk (mencegah prediksi tidak realistis). Nilai <1.0 = prediksi dibatasi di BAWAH rekor historis (konservatif, berisiko memotong prediksi wajar pada tren naik); nilai >1.0 = prediksi boleh melebihi rekor historis (lebih longgar, berisiko tidak menahan over-forecast).

**Kandidat yang diuji:** [0.8, 1.0, 1.2, 1.5]

In [5]:
hasil_cap_factor_arima = []
for nilai in [0.8, 1.0, 1.2, 1.5]:
    hasil_cap_factor_arima.append(jalankan_pipeline({'CAP_FACTOR_ARIMA': nilai}, label_eksperimen='CAP_FACTOR_ARIMA'))

df_cap_factor_arima = rangkum_dan_pilih(hasil_cap_factor_arima, 'CAP_FACTOR_ARIMA')


[EKSPERIMEN] CAP_FACTOR_ARIMA -> {'CAP_FACTOR_ARIMA': 0.8}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 166 produk dilatih dlm 148.0 detik
  LSTM: 5 klaster dilatih dlm 127.3 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,408,274 RMSE=Rp2,870,843 | LSTM: MAE=Rp1,327,530 RMSE=Rp1,587,740

[EKSPERIMEN] CAP_FACTOR_ARIMA -> {'CAP_FACTOR_ARIMA': 1.0}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


...

  ARIMA: 166 produk dilatih dlm 182.0 detik
  LSTM: 5 klaster dilatih dlm 132.1 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,524,222 RMSE=Rp3,011,125 | LSTM: MAE=Rp1,327,530 RMSE=Rp1,587,740


,CAP_FACTOR_ARIMA,jumlah_produk,bulan_testing,mae_arima,rmse_arima,waktu_arima_detik,mae_lstm,rmse_lstm,waktu_lstm_detik,rank_arima,rank_lstm,rank_gabungan
0,0.8,166,12,2.408274e+06,2.870843e+06,148.0,1.327530e+06,1.587740e+06,127.3,1.0,2.5,3.5
1,1.0,166,12,2.467634e+06,2.950422e+06,172.4,1.327530e+06,1.587740e+06,129.4,2.0,2.5,4.5
2,1.2,166,12,2.497681e+06,2.984595e+06,199.6,1.327530e+06,1.587740e+06,146.3,3.0,2.5,5.5
3,1.5,166,12,2.524222e+06,3.011125e+06,182.0,1.327530e+06,1.587740e+06,132.1,4.0,2.5,6.5


[REKOMENDASI] CAP_FACTOR_ARIMA = 0.8 memberi hasil gabungan terbaik (MAE ARIMA=Rp2,408,274, MAE LSTM=Rp1,327,530, 166 produk).


## 9. Eksperimen MIN_BULAN_AKTIF (Jumlah Bulan Cek Keaktifan Produk)

**Alasan pengujian:** Menentukan berapa bulan terakhir periode training dicek utk memastikan produk masih aktif terjual (bukan produk yang sudah discontinue). Nilai kecil = lebih longgar meloloskan produk yg jarang terjual belakangan; nilai besar = lebih ketat, berisiko membuang produk musiman yg kebetulan tidak terjual pada bulan-bulan cek tsb.

**Kandidat yang diuji:** [2, 3, 4, 6]

In [6]:
hasil_min_bulan_aktif = []
for nilai in [2, 3, 4, 6]:
    hasil_min_bulan_aktif.append(jalankan_pipeline({'MIN_BULAN_AKTIF': nilai}, label_eksperimen='MIN_BULAN_AKTIF'))

df_min_bulan_aktif = rangkum_dan_pilih(hasil_min_bulan_aktif, 'MIN_BULAN_AKTIF')


[EKSPERIMEN] MIN_BULAN_AKTIF -> {'MIN_BULAN_AKTIF': 2}
  Produk layak: 151 | Bulan training: 47 | Bulan testing: 12


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 151 produk dilatih dlm 233.2 detik
  LSTM: 5 klaster dilatih dlm 153.2 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,078,839 RMSE=Rp2,573,606 | LSTM: MAE=Rp3,648,441 RMSE=Rp4,217,258

[EKSPERIMEN] MIN_BULAN_AKTIF -> {'MIN_BULAN_AKTIF': 3}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


...

  ARIMA: 182 produk dilatih dlm 179.7 detik
  LSTM: 5 klaster dilatih dlm 139.0 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,747,421 RMSE=Rp3,217,196 | LSTM: MAE=Rp2,691,834 RMSE=Rp3,066,192


,MIN_BULAN_AKTIF,jumlah_produk,bulan_testing,mae_arima,rmse_arima,waktu_arima_detik,mae_lstm,rmse_lstm,waktu_lstm_detik,rank_arima,rank_lstm,rank_gabungan
0,2,151,12,2.078839e+06,2.573606e+06,233.2,3.648441e+06,4.217258e+06,153.2,1.0,4.0,5.0
1,3,166,12,2.467634e+06,2.950422e+06,166.5,1.327530e+06,1.587740e+06,130.6,2.0,1.0,3.0
2,4,172,12,2.505487e+06,2.960598e+06,155.4,3.506608e+06,4.190591e+06,131.3,3.0,3.0,6.0
3,6,182,12,2.747421e+06,3.217196e+06,179.7,2.691834e+06,3.066192e+06,139.0,4.0,2.0,6.0


[REKOMENDASI] MIN_BULAN_AKTIF = 3 memberi hasil gabungan terbaik (MAE ARIMA=Rp2,467,634, MAE LSTM=Rp1,327,530, 166 produk).


## Eksperimen STABILITAS RANDOM_SEED

RANDOM_SEED membuktikan hasil model TIDAK bergantung secara signifikan pada angka seed tertentu. Kita TIDAK memilih seed dgn MAE terendah (cherry-picking), melainkan melaporkan rata-rata +- std MAE lintas beberapa seed sebagai bukti kestabilan (robustness), lalu tetap memakai seed=42 sbg nilai final.

In [7]:
DAFTAR_SEED = [0, 21, 42, 84, 123]   # 42 tetap disertakan sbg pembanding

hasil_seed = []
for sd in DAFTAR_SEED:
    hasil_seed.append(jalankan_pipeline({'RANDOM_SEED': sd}, label_eksperimen='RANDOM_SEED'))

df_seed = pd.DataFrame([h for h in hasil_seed if h is not None])
display(df_seed[['RANDOM_SEED', 'mae_arima', 'rmse_arima', 'mae_lstm', 'rmse_lstm']])

print("\n--- Statistik stabilitas lintas seed ---")
print(f"ARIMA MAE : mean=Rp{df_seed['mae_arima'].mean():,.0f}  std=Rp{df_seed['mae_arima'].std():,.0f}  "
      f"CV={df_seed['mae_arima'].std()/df_seed['mae_arima'].mean()*100:.1f}%")
print(f"LSTM  MAE : mean=Rp{df_seed['mae_lstm'].mean():,.0f}  std=Rp{df_seed['mae_lstm'].std():,.0f}  "
      f"CV={df_seed['mae_lstm'].std()/df_seed['mae_lstm'].mean()*100:.1f}%")

mae_42 = df_seed[df_seed['RANDOM_SEED']==42]['mae_lstm'].values[0]
print(f"\nMAE LSTM pada seed=42: Rp{mae_42:,.0f} (dipakai sbg konfigurasi final)")
print("Interpretasi: jika CV (coefficient of variation) di bawah ~10-15%, hasil dianggap stabil")
print("lintas seed sehingga pemilihan seed=42 bukan faktor keberuntungan, melainkan nilai")
print("konvensional yang sah dipakai karena performa tidak jauh berbeda dari seed lain.")


[EKSPERIMEN] RANDOM_SEED -> {'RANDOM_SEED': 0}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 166 produk dilatih dlm 167.9 detik
  LSTM: 5 klaster dilatih dlm 153.7 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,467,634 RMSE=Rp2,950,422 | LSTM: MAE=Rp1,066,374 RMSE=Rp1,698,436

[EKSPERIMEN] RANDOM_SEED -> {'RANDOM_SEED': 21}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


...

  ARIMA: 166 produk dilatih dlm 184.6 detik
  LSTM: 5 klaster dilatih dlm 177.4 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,467,634 RMSE=Rp2,950,422 | LSTM: MAE=Rp1,649,037 RMSE=Rp2,256,594


,RANDOM_SEED,mae_arima,rmse_arima,mae_lstm,rmse_lstm
0,0,2.467634e+06,2.950422e+06,1.066374e+06,1.698436e+06
1,21,2.467634e+06,2.950422e+06,2.979722e+06,3.573056e+06
2,42,2.467634e+06,2.950422e+06,1.327530e+06,1.587740e+06
3,84,2.467634e+06,2.950422e+06,1.592551e+06,2.537905e+06
4,123,2.467634e+06,2.950422e+06,1.649037e+06,2.256594e+06



--- Statistik stabilitas lintas seed ---
ARIMA MAE : mean=Rp2,467,634  std=Rp0  CV=0.0%
LSTM  MAE : mean=Rp1,723,043  std=Rp739,830  CV=42.9%

MAE LSTM pada seed=42: Rp1,327,530 (dipakai sbg konfigurasi final)
Interpretasi: jika CV (coefficient of variation) di bawah ~10-15%, hasil dianggap stabil
lintas seed sehingga pemilihan seed=42 bukan faktor keberuntungan, melainkan nilai
konvensional yang sah dipakai karena performa tidak jauh berbeda dari seed lain.


## ABLASI GRID SEARCH AIC vs ORDER TETAP
Membuktikan apakah grid search 12 kombinasi order per produk
(dipilih via AIC) benar-benar menurunkan MAE dibanding memakai satu order tetap yg sama untuk seluruh produk (jauh lebih murah komputasinya). Fungsi ini adalah salinan jalankan_pipeline() dgn SATU perbedaan: bagian training ARIMA bisa dipaksa memakai order tetap (mode='FIXED') atau tetap grid search AIC seperti biasa (mode='AIC').

In [4]:
def jalankan_pipeline_ablasi_arima(mode, order_tetap=None, label_eksperimen=""):
    p = {**BASELINE}
    set_seed_ulang(p['RANDOM_SEED'])
    print(f"\n{'='*70}\n[ABLASI ARIMA] mode={mode} order_tetap={order_tetap}\n{'='*70}")

    df = pd.read_csv(csv_path, on_bad_lines='skip')
    df['Tanggal Pembayaran'] = pd.to_datetime(df['Tanggal Pembayaran'], format='mixed', errors='coerce')
    df = df.dropna(subset=['Tanggal Pembayaran'])
    df = df[df['Status Terakhir'] == 'Pesanan Selesai'].copy()
    for col in ['Harga Jual (IDR)','Jumlah Produk Dibeli']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['item_revenue'] = (df['Harga Jual (IDR)'] * df['Jumlah Produk Dibeli']).clip(lower=0)
    df['bulan_period'] = df['Tanggal Pembayaran'].dt.to_period('M')
    df['bulan']        = df['Tanggal Pembayaran'].dt.month
    harga_rata2 = df.groupby('Nama Produk')['Harga Jual (IDR)'].mean().to_dict()

    bulan_list  = sorted(df['bulan_period'].unique())
    split_idx   = int(len(bulan_list) * p['SPLIT_PCT'])
    bulan_train, bulan_test = bulan_list[:split_idx], bulan_list[split_idx:]

    monthly_all = (df.groupby(['bulan_period','bulan','Nama Produk'])
                     .agg(qty=('Jumlah Produk Dibeli','sum'), revenue=('item_revenue','sum'))
                     .reset_index().sort_values(['Nama Produk','bulan_period']))
    monthly_train = monthly_all[monthly_all['bulan_period'].isin(bulan_train)]
    monthly_test  = monthly_all[monthly_all['bulan_period'].isin(bulan_test)]

    iqr_bounds = {}
    for prod in monthly_train['Nama Produk'].unique():
        vals = monthly_train[monthly_train['Nama Produk']==prod]['qty']
        if len(vals) < 4: continue
        Q1, Q3 = vals.quantile(0.25), vals.quantile(0.75); IQR = Q3 - Q1
        iqr_bounds[prod] = (max(0.0, Q1-p['IQR_MULTIPLIER']*IQR), Q3+p['IQR_MULTIPLIER']*IQR)
    def _clip_qty(row):
        b = iqr_bounds.get(row['Nama Produk'])
        return row['qty'] if b is None else float(np.clip(row['qty'], b[0], b[1]))
    monthly_all['qty'] = monthly_all.apply(_clip_qty, axis=1)
    monthly_train = monthly_all[monthly_all['bulan_period'].isin(bulan_train)]
    monthly_test  = monthly_all[monthly_all['bulan_period'].isin(bulan_test)]

    produk_count = monthly_train.groupby('Nama Produk')['bulan_period'].count()
    produk_layak_awal = produk_count[produk_count >= p['MIN_BULAN']].index.tolist()
    bulan_train_bersih = [b for b in bulan_train if str(b) not in BULAN_ANOMALI]
    bulan_cek_aktif = bulan_train_bersih[-p['MIN_BULAN_AKTIF']:]
    def cek_aktif(prod):
        df_p = monthly_train[monthly_train['Nama Produk']==prod]
        return df_p[df_p['bulan_period'].isin(bulan_cek_aktif)]['qty'].sum() > 0
    produk_layak = [prod for prod in produk_layak_awal if cek_aktif(prod)]

    mean_qty_produk, median_qty_produk, max_qty_produk = {}, {}, {}
    for prod in produk_layak:
        vals = monthly_train[(monthly_train['Nama Produk']==prod) & (~monthly_train['bulan_period'].astype(str).isin(BULAN_ANOMALI))]['qty'].values
        mean_qty_produk[prod]   = max(float(vals.mean()), 1.0) if len(vals)>0 else 1.0
        median_qty_produk[prod] = max(float(np.median(vals)), 1.0) if len(vals)>0 else 1.0
        max_qty_produk[prod]    = max(float(vals.max()), 1.0) if len(vals)>0 else 1.0

    # --- Training ARIMA: mode AIC (grid search) vs mode FIXED (order tetap) ---
    t0 = time.time()
    best_orders_arima, bias_correction = {}, {}
    n_kombinasi_dicoba = 0
    for produk in produk_layak:
        df_c = monthly_train[monthly_train['Nama Produk']==produk].sort_values('bulan_period')
        df_c = df_c[~df_c['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)].tail(p['N_WINDOW_ARIMA'])
        if len(df_c) < 10:
            bias_correction[produk] = 1.0
            continue
        fl_v = df_c['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x,1.0)).values
        ts_log = np.log1p(df_c['qty'].clip(lower=0.1).values / np.maximum(fl_v, 0.1))

        if mode == 'FIXED':
            best_order = order_tetap
            n_kombinasi_dicoba += 1
        else:  # mode == 'AIC'
            best_aic, best_order = np.inf, (1,1,1)
            for order in ARIMA_ORDERS:
                try:
                    m = ARIMA(ts_log, order=order).fit()
                    n_kombinasi_dicoba += 1
                    if m.aic < best_aic:
                        best_aic, best_order = m.aic, order
                except Exception:
                    pass
        best_orders_arima[produk] = best_order
        bias_correction[produk] = 1.0
        if len(df_c) > p['BIAS_HOLDOUT'] + 8:
            try:
                tr_b, ho_b = df_c.iloc[:-p['BIAS_HOLDOUT']], df_c.iloc[-p['BIAS_HOLDOUT']:]
                ts_tb = np.log1p(tr_b['qty'].clip(lower=0.1).values / np.maximum(
                    tr_b['bulan_period'].astype(str).map(lambda x:KALENDER_LIBUR.get(x,1.0)).values, 0.1))
                m_b = ARIMA(ts_tb, order=best_order).fit()
                fc_log = np.clip(m_b.forecast(steps=p['BIAS_HOLDOUT']), -2.0, 10.0)
                fl_hb  = ho_b['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x,1.0)).values
                fc_qty = np.expm1(fc_log) * fl_hb
                rasio  = max(float(np.asarray(fc_qty).sum()), 1e-6) / float(ho_b['qty'].values.sum())
                bias_correction[produk] = float(np.clip(rasio, 0.7, 1.5))
            except Exception:
                bias_correction[produk] = 1.0
    t_arima = time.time() - t0

    log_bulan = []
    for bulan_pred in bulan_test:
        avail = monthly_all[monthly_all['bulan_period'] < bulan_pred]
        df_a = prediksi_arima(p, bulan_pred, avail, produk_layak, best_orders_arima, bias_correction, harga_rata2, max_qty_produk)
        aktual_b = monthly_test[monthly_test['bulan_period']==bulan_pred][['Nama Produk','qty','revenue']].rename(
            columns={'Nama Produk':'nama_produk','qty':'aktual_qty','revenue':'aktual_rev'})
        df_m = pd.merge(df_a, aktual_b, on='nama_produk', how='left').fillna(0)
        log_bulan.append({'aktual': df_m['aktual_rev'].sum(), 'pred_arima': df_m['pred_rev_arima'].sum()})

    df_log = pd.DataFrame(log_bulan)
    mae_a  = float(mean_absolute_error(df_log['aktual'], df_log['pred_arima']))
    rmse_a = float(np.sqrt(mean_squared_error(df_log['aktual'], df_log['pred_arima'])))
    print(f"  ARIMA ({mode}) -> MAE=Rp{mae_a:,.0f} RMSE=Rp{rmse_a:,.0f} | "
          f"{n_kombinasi_dicoba} fitting SARIMAX dijalankan | waktu={t_arima:.1f} detik")
    return {'mode': label_eksperimen, 'mae_arima': mae_a, 'rmse_arima': rmse_a,
            'n_fitting': n_kombinasi_dicoba, 'waktu_detik': round(t_arima,1)}

# --- Jalankan perbandingan ---
hasil_aic = jalankan_pipeline_ablasi_arima('AIC', label_eksperimen='Grid Search AIC (baseline)')
hasil_fixed_110 = jalankan_pipeline_ablasi_arima('FIXED', order_tetap=(1,1,0), label_eksperimen='Order tetap (1,1,0)')
hasil_fixed_111 = jalankan_pipeline_ablasi_arima('FIXED', order_tetap=(1,1,1), label_eksperimen='Order tetap (1,1,1)')

df_ablasi_aic = pd.DataFrame([hasil_aic, hasil_fixed_110, hasil_fixed_111])
display(df_ablasi_aic)
selisih_pct = (df_ablasi_aic[df_ablasi_aic['mode'].str.contains('tetap')]['mae_arima'].min() - hasil_aic['mae_arima']) / hasil_aic['mae_arima'] * 100
print(f"\nJika grid search AIC menghasilkan MAE lebih rendah drpd SEMUA order tetap, ini membuktikan")
print(f"proses grid search memang berkontribusi nyata (selisih ~{selisih_pct:.1f}% vs order tetap terbaik),")
print(f"meski butuh {hasil_aic['n_fitting']}x lebih banyak proses fitting ({hasil_aic['n_fitting']} vs {hasil_fixed_110['n_fitting']} fitting).")


[ABLASI ARIMA] mode=AIC order_tetap=None


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA (AIC) -> MAE=Rp2,467,634 RMSE=Rp2,950,422 | 1992 fitting SARIMAX dijalankan | waktu=1725.4 detik

[ABLASI ARIMA] mode=FIXED order_tetap=(1, 1, 0)


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  ARIMA (FIXED) -> MAE=Rp3,040,924 RMSE=Rp3,468,323 | 166 fitting SARIMAX dijalankan | waktu=6.7 detik

[ABLASI ARIMA] mode=FIXED order_tetap=(1, 1, 1)


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA (FIXED) -> MAE=Rp2,452,351 RMSE=Rp2,940,528 | 166 fitting SARIMAX dijalankan | waktu=56.9 detik


,mode,mae_arima,rmse_arima,n_fitting,waktu_detik
0,Grid Search AIC (baseline),2.467634e+06,2.950422e+06,1992,1725.4
1,"Order tetap (1,1,0)",3.040924e+06,3.468323e+06,166,6.7
2,"Order tetap (1,1,1)",2.452351e+06,2.940528e+06,166,56.9



Jika grid search AIC menghasilkan MAE lebih rendah drpd SEMUA order tetap, ini membuktikan
proses grid search memang berkontribusi nyata (selisih ~-0.6% vs order tetap terbaik),
meski butuh 1992x lebih banyak proses fitting (1992 vs 166 fitting).


## CAP_FACTOR_LSTM (Batas Atas Prediksi LSTM)
CAP_FACTOR_LSTM membatasi prediksi LSTM relatif thd rekor qty tertinggi historis produk (mencegah prediksi tidak realistis akibat ekstrapolasi model). Nilai baseline 1.5 dipilih lebih longgar dari CAP_FACTOR_ARIMA=1.0 karena LSTM dianggap lebih mampu menangkap tren naik yang sah (bukan noise), tapi asumsi ini belum pernah diuji langsung -- nilai terlalu ketat (1.0) berisiko memotong prediksi wajar pada tren naik tajam; nilai terlalu longgar (2.5) berisiko tidak menahan prediksi ekstrem yang tidak realistis.

In [5]:
hasil_cap_lstm = []
for nilai in [1.0, 1.2, 1.5, 1.8, 2.0, 2.5]:
    hasil_cap_lstm.append(jalankan_pipeline({'CAP_FACTOR_LSTM': nilai}, label_eksperimen='CAP_FACTOR_LSTM'))

df_cap_lstm = rangkum_dan_pilih(hasil_cap_lstm, 'CAP_FACTOR_LSTM')


[EKSPERIMEN] CAP_FACTOR_LSTM -> {'CAP_FACTOR_LSTM': 1.0}
  Produk layak: 166 | Bulan training: 47 | Bulan testing: 12


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  ARIMA: 166 produk dilatih dlm 817.7 detik
  LSTM: 5 klaster dilatih dlm 235.2 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

...

  ARIMA: 166 produk dilatih dlm 141.4 detik
  LSTM: 5 klaster dilatih dlm 121.0 detik


c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (6)\backend\.venv\lib\site-packages\statsmodels\base\

  HASIL -> ARIMA: MAE=Rp2,467,634 RMSE=Rp2,950,422 | LSTM: MAE=Rp1,327,530 RMSE=Rp1,587,740


,CAP_FACTOR_LSTM,jumlah_produk,bulan_testing,mae_arima,rmse_arima,waktu_arima_detik,mae_lstm,rmse_lstm,waktu_lstm_detik,rank_arima,rank_lstm,rank_gabungan
0,1.0,166,12,2.467634e+06,2.950422e+06,817.7,1.327530e+06,1.587740e+06,235.2,3.5,3.5,7.0
1,1.2,166,12,2.467634e+06,2.950422e+06,754.7,1.327530e+06,1.587740e+06,255.3,3.5,3.5,7.0
2,1.5,166,12,2.467634e+06,2.950422e+06,800.9,1.327530e+06,1.587740e+06,152.6,3.5,3.5,7.0
3,1.8,166,12,2.467634e+06,2.950422e+06,165.3,1.327530e+06,1.587740e+06,118.7,3.5,3.5,7.0
4,2.0,166,12,2.467634e+06,2.950422e+06,167.4,1.327530e+06,1.587740e+06,115.4,3.5,3.5,7.0
5,2.5,166,12,2.467634e+06,2.950422e+06,141.4,1.327530e+06,1.587740e+06,121.0,3.5,3.5,7.0


[REKOMENDASI] CAP_FACTOR_LSTM = 1.0 memberi hasil gabungan terbaik (MAE ARIMA=Rp2,467,634, MAE LSTM=Rp1,327,530, 166 produk).


## SEL PENUTUP -- RINGKASAN SELURUH EKSPERIMEN


In [6]:
# --- Bagian 1: 9 parameter numerik OFAT + CAP_FACTOR_LSTM (10 total) ---
semua_df = {
    'SPLIT_PCT': df_split_pct, 'MIN_BULAN': df_min_bulan, 'N_WINDOW_ARIMA': df_n_window_arima,
    'SEQ_LEN': df_seq_len, 'N_CLUSTER': df_n_cluster, 'IQR_MULTIPLIER': df_iqr_multiplier,
    'BIAS_HOLDOUT': df_bias_holdout, 'CAP_FACTOR_ARIMA': df_cap_factor_arima,
    'MIN_BULAN_AKTIF': df_min_bulan_aktif,
    'CAP_FACTOR_LSTM': df_cap_lstm,   # <-- eksperimen tambahan C
}

konfigurasi_final = {}
ringkasan_rows = []
for nama_param, df_hasil in semua_df.items():
    if df_hasil is None or df_hasil.empty:
        continue
    terbaik = df_hasil.loc[df_hasil['rank_gabungan'].idxmin()]
    konfigurasi_final[nama_param] = terbaik[nama_param]
    ringkasan_rows.append({
        'Parameter': nama_param, 'Nilai Terpilih': terbaik[nama_param],
        'MAE ARIMA (Rp)': f"{terbaik['mae_arima']:,.0f}", 'MAE LSTM (Rp)': f"{terbaik['mae_lstm']:,.0f}",
        'Metode Pemilihan': 'MAE terendah (OFAT)',
    })

# --- Bagian 2: RANDOM_SEED -- stabilitas, BUKAN cherry-pick ---
if 'df_seed' in dir() and df_seed is not None and not df_seed.empty:
    cv_lstm = df_seed['mae_lstm'].std() / df_seed['mae_lstm'].mean() * 100
    status_stabil = "STABIL" if cv_lstm < 15 else "PERLU DIPERHATIKAN (variansi tinggi antar-seed)"
    konfigurasi_final['RANDOM_SEED'] = 42  # tetap nilai konvensional, bukan hasil pemilihan
    mae_lstm_42 = df_seed[df_seed['RANDOM_SEED'] == 42]['mae_lstm'].values[0]
    mae_arima_42 = df_seed[df_seed['RANDOM_SEED'] == 42]['mae_arima'].values[0]
    ringkasan_rows.append({
        'Parameter': 'RANDOM_SEED', 'Nilai Terpilih': 42,
        'MAE ARIMA (Rp)': f"{mae_arima_42:,.0f}", 'MAE LSTM (Rp)': f"{mae_lstm_42:,.0f}",
        'Metode Pemilihan': f'Konvensional -- CV lintas seed={cv_lstm:.1f}% ({status_stabil}), bukan MAE terendah',
    })

# --- Bagian 3: Metode ARIMA -- AIC grid search vs order tetap ---
if 'df_ablasi_aic' in dir() and df_ablasi_aic is not None and not df_ablasi_aic.empty:
    mae_aic = df_ablasi_aic[df_ablasi_aic['mode'].str.contains('AIC')]['mae_arima'].values[0]
    baris_fixed_terbaik = df_ablasi_aic[df_ablasi_aic['mode'].str.contains('tetap')].loc[
        df_ablasi_aic[df_ablasi_aic['mode'].str.contains('tetap')]['mae_arima'].idxmin()]
    selisih_pct = (baris_fixed_terbaik['mae_arima'] - mae_aic) / mae_aic * 100
    if selisih_pct > 5:
        metode_final, alasan = 'Grid Search AIC', f'MAE {selisih_pct:.1f}% lebih rendah drpd order tetap terbaik -- worth biaya komputasi tambahan'
    else:
        metode_final, alasan = 'Order Tetap ' + baris_fixed_terbaik['mode'], f'Selisih MAE vs AIC hanya {selisih_pct:.1f}% -- grid search tidak worth biaya komputasi {df_ablasi_aic[df_ablasi_aic["mode"].str.contains("AIC")]["n_fitting"].values[0]}x lebih mahal'
    konfigurasi_final['METODE_ARIMA'] = metode_final
    ringkasan_rows.append({
        'Parameter': 'METODE_ARIMA (AIC vs Fixed)', 'Nilai Terpilih': metode_final,
        'MAE ARIMA (Rp)': f"{min(mae_aic, baris_fixed_terbaik['mae_arima']):,.0f}", 'MAE LSTM (Rp)': '-',
        'Metode Pemilihan': alasan,
    })

# --- Tampilkan ringkasan lengkap ---
print("Konfigurasi final hasil SELURUH eksperimen (9 parameter OFAT + 3 tambahan):")
display(pd.DataFrame(ringkasan_rows))
print()
print(konfigurasi_final)
print()
print("Catatan metodologis:")
print("1. Karena eksperimen ini bersifat OFAT (satu parameter per waktu, bukan grid penuh),")
print("   konfigurasi final adalah PENDEKATAN TERBAIK berbasis bukti, bukan jaminan optimum global.")
print("2. RANDOM_SEED sengaja TIDAK dipilih berdasarkan MAE terendah -- nilai 42 dipertahankan")
print("   selama variansi (CV) antar-seed rendah, sesuai prinsip bahwa seed bukan hyperparameter")
print("   yang boleh di-tuning terhadap hasil (cherry-picking).")
print("3. Pemilihan metode ARIMA (AIC vs order tetap) mempertimbangkan trade-off akurasi VS biaya")
print("   komputasi, bukan MAE semata -- keputusan arsitektur pipeline, bukan tuning nilai kontinu.")
print("4. Kombinasi seluruh parameter terpilih tetap perlu divalidasi ulang dengan SATU kali training")
print("   penuh memakai seluruh nilai final sekaligus, sebelum dipakai sebagai hasil akhir Bab 4.")

NameError: name 'df_split_pct' is not defined